In [ ]:
from dotenv import load_dotenv

load_dotenv() 

# Card

- works well

In [ ]:
from typing import Optional

from langchain_core.pydantic_v1 import BaseModel, Field
from vizro.models import Card

In [ ]:
from typing import Optional

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_openai import ChatOpenAI

# Define a custom prompt to provide instructions and any additional context.
# 1) You can add examples into the prompt template to improve extraction quality
# 2) Introduce additional parameters to take context into account (e.g., include metadata
#    about the document from which the text was extracted.)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert extraction algorithm. "
            "Only extract relevant information from the text. "
            "If you do not know the value of an attribute asked to extract, "
            "return null for the attribute's value.",
        ),
        # Please see the how-to about improving performance with
        # reference examples.
        # MessagesPlaceholder('examples'),
        ("human", "{text}"),
    ]
)

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    temperature=0,
    model="gpt-3.5-turbo",
    )

runnable = prompt | model.with_structured_output(schema=Card)

In [ ]:
dataset_schema= """ Here's how the data schema looks like: total_bill: Numeric tip: Numeric sex: Categorical (Male/Female) smoker: Categorical (Yes/No) day: Categorical (Sun, Mon, Tue, Wed, Thu, Fri, Sat) time: Categorical (Lunch, Dinner) size: Numeric """ 

code_problem = """ I want to create a Vizro style dashboard. I need One page. On this page, there are two charts. The first bar chart shows the average <tip> per <day>. The second chart shows the distribution of <total_bill>. add a <Checklist> filter to filter on <time>. On this page, also add a <Card> with text 'This page is using the tips dataset.' """
runnable.invoke({"text": dataset_schema+code_problem})

# Dashboard and Page

- not working
- error message: BadRequestError: Error code: 400 - {'error': {'message': "Invalid schema for function 'Dashboard': [{'title': 'Component Id', 'type': 'string'}, {'title': 'Component Property', 'type': 'string'}] is not of type 'object', 'boolean'.", 'type': 'invalid_request_error', 'param': 'tools[0].function.parameters', 'code': 'invalid_function_parameters'}}

#### reason: schema conflict?
#### can subclassing Dashboard solve the error?

In [ ]:
from vizro.models import Dashboard

In [ ]:
runnable = prompt | model.with_structured_output(schema=Dashboard)
runnable.invoke({"text": dataset_schema+code_problem})

In [ ]:
from vizro.models import Page

runnable = prompt | model.with_structured_output(schema=Page)
runnable.invoke({"text": dataset_schema+code_problem})

# Filter

- works when no target
- error when setting target. error message: Target bar_chart not found in model_manager

In [ ]:
from vizro.models import Filter

runnable = prompt | model.with_structured_output(schema=Filter)
runnable.invoke({"text": dataset_schema+code_problem})

In [ ]:
data_schema = """
Here's how the data schema looks like:
country: Name of the country (categorical).
continent: Continent to which the country belongs (categorical).
year: Year for which the data is recorded (numeric).
lifeExp: Life expectancy of the population (numeric).
pop: Population of the country (numeric).
gdpPercap: GDP per capita of the country (numeric).
"""

query = ("I need a page with a bar chart shoing the population per continent "
         "and a scatter chart showing the life expectency per country as a function gdp. "
         "Make a filter on the country column and use a dropdown as selector. This filter should only "
         "apply to the bar chart. The bar chart should be a stacked bar chart, while "
         "the scatter chart should be colored by the column 'continent'. Make another filter "
         "that filters the GDP column, and it only applies to the scatter chart. I also want "
         "a table that shows the data. The title of the page should be: `This is big time data`. I also want a second page with two "
         "cards on it. One should be a card saying: `This was the jolly data dashboard, it was created in Vizro which is amazing`"
         ", and the second card should refer to the "
         " documentation and link to `https://vizro.readthedocs.io/`. The title of the dashboard should be: `My wonderful "
         "jolly dashboard showing a lot of data`.")

In [ ]:
from vizro.models import Filter

runnable = prompt | model.with_structured_output(schema=Filter)
runnable.invoke({"text": data_schema+query})

# AgGrid and Graph

- not working directly

In [ ]:
from vizro.models import AgGrid

runnable = prompt | model.with_structured_output(schema=AgGrid)
runnable.invoke({"text": data_schema+query})

In [ ]:
from vizro.models import Graph

runnable = prompt | model.with_structured_output(schema=Graph)
runnable.invoke({"text": data_schema+query})

# Layout

- might need RAG

In [ ]:
question = "How can I build a vizro dashboard with iris dataset, with 3 scatter plot, Sepal width with length, Petal width with length, \
    and Petal Area with Sepal Area. use layout grid '[[0,1],[0,2]]' also include a filter, \
    please only use import vizro.plotly.express as px,  \
        DO NOT use import plotly.express as px"

In [ ]:
from vizro.models import Layout

runnable = prompt | model.with_structured_output(schema=Layout)
runnable.invoke({"text": question})

In [ ]:
question_without_exact_grid = "How can I build a vizro dashboard with iris dataset, with 3 scatter plot, Sepal width with length, Petal width with length, \
    and Petal Area with Sepal Area. Layout these 3 plots as one on the left and the other two stack on the right. also include a filter, \
    please only use import vizro.plotly.express as px,  \
        DO NOT use import plotly.express as px"

In [ ]:
from vizro.models import Layout

runnable = prompt | model.with_structured_output(schema=Layout)

runnable.invoke({"text": question_without_exact_grid})